In [1]:
import os
import json
import re
from transformers import AutoTokenizer

In [2]:
my_cache_dir = r"E:\Models"
model_name="Qwen/Qwen2.5-3B"
BLOCK_SIZE = 1024  
OVERLAP = 128
STRIDE = BLOCK_SIZE - OVERLAP
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",cache_dir=my_cache_dir)

In [3]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
def tokenize_fn(examples):
    tokens = tokenizer(examples["text"],truncation=True,padding="max_length",max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [4]:
file_directories = []
chunks = []

In [5]:

schad_comp_base = "./docs/ui/apps/v4/content/docs/components/base"
schadcn_base_source = "./docs/ui/apps/v4/examples/base"

for comp in os.listdir(schad_comp_base):
    if ".mdx" in comp:
        file_directories.append(f"{schad_comp_base}/{comp}")
        
for source in os.listdir(schadcn_base_source):
    if ".tsx" in source:
            file_directories.append(f"{schadcn_base_source}/{source}")
                
len(file_directories)

577

In [6]:
def clean_mdx_content(text):
    """Strips out messy React import headers from the top of MDX files."""
    text = re.sub(r'^import\s+.*?\s+from\s+[\'"].*?[\'"];?\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^export\s+default\s+.*$', '', text, flags=re.MULTILINE)
    return text.strip()

In [7]:
for file in file_directories:
    with open(file,"r", encoding="utf-8") as f:
        f_name = os.path.basename(f.name)
        if ".mdx" in f_name:
            purpose = f_name.removesuffix(".mdx")
            body = clean_mdx_content(f.read())
        else:
            purpose = f_name.removesuffix(".tsx")
            body = f.read().strip()
        header = "Framework: NextJs-Ui helper(by shadcn-ui) \n Purpose:" + purpose
        full_text = header+ "\n" + "Content: \n" + body
        tokens = tokenizer.encode(full_text, add_special_tokens=False)
        
        for i in range(0, len(tokens), STRIDE):
            chunk_tokens = tokens[i:i + BLOCK_SIZE]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            chunks.append({
                "text": chunk_text,
                "framework": "NextJS- Schadcn Ui",
                "purpose": purpose,
                "source": f_name,
            })
        
output_file = "chunks/schadcn.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"Success! Generated {len(chunks)} chunks in {output_file}")
        

Success! Generated 782 chunks in chunks/schadcn.jsonl
